In [ ]:
# In[1]:

import pandas as pd

# 1) Load metric.csv and parse timestamps as UTC
metric_df = pd.read_csv("metric.csv")
metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# 1) List distinct (cmdb_id, kpi_name) pairs and counts (top 50 by count)
pairs_count = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
pairs_top50 = pairs_count.head(50)

# 2) Compute global P95 per (cmdb_id, kpi_name) using full series (before any filtering)
global_p95 = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_P95')
)

# Merge counts and global P95 for a base table
pairs_base = pairs_count.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# 3) Filter metric points to the incident window (UTC)
start_ts = pd.to_datetime("2024-01-20 22:55:32", utc=True)
end_ts = pd.to_datetime("2024-01-20 23:25:32", utc=True)
window_df = metric_df[(metric_df['ts'] >= start_ts) & (metric_df['ts'] <= end_ts)].copy()

# number_of_points_in_window per pair
window_counts = (
    window_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .size()
    .reset_index(name='number_of_points_in_window')
)

# For points above global P95: join window_df with global_p95, then filter value > global_P95
window_with_p95 = window_df.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')
above_mask = window_with_p95['value'] > window_with_p95['global_P95']
above_df = window_with_p95[above_mask]

# number_of_points_above_P95 and earliest timestamp of such point per pair
above_agg = (
    above_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(number_of_points_above_P95=('value', 'size'),
         earliest_ts_above=('ts', 'min'))
    .reset_index()
)

# Convert earliest timestamp to ISO UTC string, keep nulls as None
def to_iso_or_none(ts):
    if pd.isna(ts):
        return None
    return ts.isoformat()

above_agg['earliest_timestamp_of_point_above_P95'] = above_agg['earliest_ts_above'].apply(to_iso_or_none)
above_agg = above_agg.drop(columns=['earliest_ts_above'])

# 3) Build final report by merging base pairs with window counts and above_agg
report = (
    pairs_base
    .merge(window_counts, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(above_agg, on=['cmdb_id', 'kpi_name'], how='left')
)

# Fill NaNs for counts with 0 and keep earliest timestamp null when missing
report['number_of_points_in_window'] = report['number_of_points_in_window'].fillna(0).astype(int)
report['number_of_points_above_P95'] = report['number_of_points_above_P95'].fillna(0).astype(int)
report['earliest_timestamp_of_point_above_P95'] = report['earliest_timestamp_of_point_above_P95'].where(
    report['earliest_timestamp_of_point_above_P95'].notna(), None)

# Select columns required for output
report_selected = report[[
    'cmdb_id',
    'kpi_name',
    'global_P95',
    'number_of_points_in_window',
    'number_of_points_above_P95',
    'earliest_timestamp_of_point_above_P95'
]]

# 4) Return at most top 20 rows where number_of_points_above_P95 > 0, sorted descending by that count
anomalies_top20 = (
    report_selected[report_selected['number_of_points_above_P95'] > 0]
    .sort_values('number_of_points_above_P95', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# Also return a small table listing all unique cmdb_id values found in metric.csv
unique_cmdbs = pd.DataFrame({'cmdb_id': sorted(metric_df['cmdb_id'].dropna().unique())})

# Keep full intermediate tables in variables (pairs_count, global_p95, window_df, above_df, report) if needed,
# but only display compact outputs as requested:
pairs_top50, anomalies_top20, unique_cmdbs

```
Out[1]:
```
```python
# Summarize anomalies found in previous analysis (uses variables in kernel: anomalies_top20, unique_cmdbs)
if 'anomalies_top20' not in globals():
    summary = "No analysis results (anomalies_top20 not found in kernel)."
else:
    total_unique_cmdb = int(unique_cmdbs.shape[0]) if 'unique_cmdbs' in globals() else None
    total_anomaly_rows = int(anomalies_top20.shape[0])
    total_anomaly_points = int(anomalies_top20['number_of_points_above_P95'].sum()) if total_anomaly_rows>0 else 0

    if total_anomaly_rows == 0:
        summary = (
            f"Metric summary: {total_unique_cmdb} unique services/components found. "
            "No metric points in the incident window exceeded their global P95."
        )
    else:
        # Top components by total anomaly points
        comp_agg = (
            anomalies_top20
            .groupby('cmdb_id', as_index=False)['number_of_points_above_P95']
            .sum()
            .sort_values('number_of_points_above_P95', ascending=False)
        )
        top_comps = comp_agg.head(5)
        top_list = []
        for _, row in top_comps.iterrows():
            comp = row['cmdb_id']
            pts = int(row['number_of_points_above_P95'])
            # earliest anomaly timestamp for this component
            earliest = anomalies_top20[anomalies_top20['cmdb_id']==comp]['earliest_timestamp_of_point_above_P95'].min()
            top_list.append(f"{comp}: {pts} point(s) (earliest {earliest})")

        # Example top anomalous KPI rows (limit to 5) to show what KPIs were involved
        example_rows = anomalies_top20[['cmdb_id','kpi_name','number_of_points_above_P95','earliest_timestamp_of_point_above_P95']].head(5)
        example_lines = "; ".join(
            f"{r['cmdb_id']}/{r['kpi_name']}={r['number_of_points_above_P95']}@{r['earliest_timestamp_of_point_above_P95']}"
            for _, r in example_rows.iterrows()
        )

        summary = (
            f"Metric summary: {total_unique_cmdb} unique services/components found. "
            f"{total_anomaly_rows} (cmdb_id, kpi) pairs had points above their global P95 within the incident window, "
            f"totaling {total_anomaly_points} anomalous point(s). "
            f"Top affected components: {', '.join(top_list)}. "
            f"Example anomalous KPIs: {example_lines}."
        )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  count
0          carts         cpu     25
1          carts      diskio     25
2          carts  latency-50     25
3          carts  latency-90     25
4          carts         mem     25
..           ...         ...    ...
45  queue-master         mem     25
46  queue-master      socket     25
47      rabbitmq         cpu     25
48      rabbitmq      diskio     25
49      rabbitmq         mem     25

[50 rows x 3 columns],          cmdb_id    kpi_name    global_P95  number_of_points_in_window  number_of_points_above_P95 earliest_timestamp_of_point_above_P95
0          carts         cpu  4.298539e+00                          25                           2             2024-01-20T22:58:00+00:00
1          carts  latency-50  1.613659e-02                          25                           2             2024-01-20T23:10:00+00:00
2          carts  latency-90  3.170315e-02                          25                           2             2024-01-20T23:19:00+00:00
3          carts         mem  2.064875e+08                          25                           2             2024-01-20T23:14:00+00:00
4          carts      socket  1.039667e+01                          25                           2             2024-01-20T22:58:00+00:00
5          carts    workload  8.534613e+00                          25                           2             2024-01-20T23:00:00+00:00
6       carts-db         cpu  3.559212e+00                          25                           2             2024-01-20T23:08:00+00:00
7       carts-db      diskio  2.112777e+06                          25                           2             2024-01-20T23:07:00+00:00
8       carts-db         mem  8.053645e+07                          25                           2             2024-01-20T23:21:00+00:00
9      catalogue         cpu  1.971724e-01                          25                           2             2024-01-20T22:59:00+00:00
10     catalogue  latency-50  3.128734e-03                          25                           2             2024-01-20T22:58:00+00:00
11     catalogue  latency-90  4.837634e-03                          25                           2             2024-01-20T23:00:00+00:00
12     catalogue         mem  6.132701e+06                          25                           2             2024-01-20T23:21:00+00:00
13     catalogue    workload  4.257483e+00                          25                           2             2024-01-20T23:02:00+00:00
14  catalogue-db         cpu  2.279101e-01                          25                           2             2024-01-20T22:58:00+00:00
15  catalogue-db         mem  4.308820e+08                          25                           2             2024-01-20T23:20:00+00:00
16     front-end         cpu  4.979593e+00                          25                           2             2024-01-20T23:02:00+00:00
17     front-end  latency-50  4.073165e-02                          25                           2             2024-01-20T23:19:00+00:00
18     front-end  latency-90  9.533064e-01                          25                           2             2024-01-20T23:19:00+00:00
19     front-end         mem  1.065863e+08                          25                           2             2024-01-20T23:15:00+00:00,               cmdb_id
0               carts
1            carts-db
2           catalogue
3        catalogue-db
4           front-end
5              orders
6           orders-db
7             payment
8        queue-master
9            rabbitmq
10  rabbitmq-exporter
11         session-db
12           shipping
13               user
14            user-db)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse metric_df if present, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure timestamp column parsed where needed; create ts for full df to reuse
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# 1) Recompute global P95 for every (cmdb_id, kpi_name) using full series
global_p95 = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_P95')
)

# 2) Filter metric.csv to the incident window (UTC)
start_ts = pd.to_datetime("2024-01-20 22:55:32", utc=True)
end_ts = pd.to_datetime("2024-01-20 23:25:32", utc=True)
window_df = metric_df[(metric_df['ts'] >= start_ts) & (metric_df['ts'] <= end_ts)].copy()

# 3) For each pair present in the filtered window, mark anomalous points where value > global_P95
window_df = window_df.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')
window_df['anomaly'] = window_df['value'] > window_df['global_P95']

# 4) Aggregate per (cmdb_id, kpi_name)
def compute_group_agg(g):
    cmdb = g.name[0]
    kpi = g.name[1]
    gp95 = g['global_P95'].iloc[0]  # same for group
    num_in_window = len(g)
    num_above = int(g['anomaly'].sum())
    max_val = g['value'].max()
    # longest consecutive anomalous run (in points/minutes)
    if num_above == 0:
        longest_run = 0
        first_iso = None
        last_iso = None
    else:
        # get timestamps of anomalous points sorted
        at = g.loc[g['anomaly'], 'ts'].sort_values()
        # compute diffs between successive anomalous timestamps
        diffs = at.diff()
        # new run starts where diff != 1 minute (or diff is NaT for first)
        run_id = (diffs != pd.Timedelta(minutes=1)).cumsum()
        run_lengths = at.groupby(run_id).size()
        longest_run = int(run_lengths.max())
        first_iso = at.min().isoformat()
        last_iso = at.max().isoformat()
    # max over threshold ratio
    if gp95 == 0 or pd.isna(gp95):
        ratio = float('inf') if (max_val > gp95) else 0.0
    else:
        ratio = (max_val - gp95) / gp95
    return pd.Series({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'global_P95': gp95,
        'number_of_points_in_window': num_in_window,
        'number_of_points_above_P95': num_above,
        'longest_consecutive_anomalous_run': longest_run,
        'first_anomalous_timestamp': first_iso,
        'last_anomalous_timestamp': last_iso,
        'max_value_in_window': max_val,
        'max_over_threshold_ratio': ratio
    })

grouped = window_df.groupby(['cmdb_id', 'kpi_name'], sort=False)
agg_df = grouped.apply(compute_group_agg).reset_index(drop=True)

# 5) Return up to top 20 rows where number_of_points_above_P95>0,
# sorted by longest_consecutive_anomalous_run desc, then by number_of_points_above_P95 desc.
anomalies_summary = agg_df[agg_df['number_of_points_above_P95'] > 0].copy()
anomalies_summary = anomalies_summary.sort_values(
    ['longest_consecutive_anomalous_run', 'number_of_points_above_P95'],
    ascending=[False, False]
).reset_index(drop=True)

anomalies_summary_top20 = anomalies_summary.head(20)

# Also return a separate small list of distinct cmdb_id values that had any anomalous points in the window
anomalous_cmdbs = pd.DataFrame({'cmdb_id': sorted(anomalies_summary['cmdb_id'].unique())})

# Keep full intermediate tables in kernel variables: global_p95, window_df, agg_df, anomalies_summary
# Display only the requested compact outputs:
anomalies_summary_top20, anomalous_cmdbs

```
Out[2]:
```
```python
summary = (
    "Incident window: 2024-01-20 22:55:32 to 2024-01-20 23:25:32 UTC.\n"
    "Top 20 (cmdb_id, kpi) pairs had anomalous points above their global P95 during the window. "
    "Most displayed pairs had 2 anomalous points and the longest consecutive anomalous runs were typically 2 minutes.\n"
    "Notable spikes: carts/cpu showed a large spike (max_over_threshold_ratio ≈ 2.83, max ≈ 16.48 vs P95 ≈ 4.30), "
    "user/diskio (ratio ≈ 2.22), and queue-master/diskio (ratio ≈ 0.93). "
    "Many other anomalies were small deviations above P95 (mem and latency KPIs with ratios ≪ 0.1).\n"
    "Components with any anomalous metric in the window: carts, carts-db, catalogue, catalogue-db, front-end, orders, "
    "orders-db, payment, queue-master, rabbitmq, rabbitmq-exporter, session-db, shipping, user, user-db."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name    global_P95  number_of_points_in_window  number_of_points_above_P95  longest_consecutive_anomalous_run  first_anomalous_timestamp   last_anomalous_timestamp  max_value_in_window  max_over_threshold_ratio
0          carts  latency-90  3.170315e-02                          25                           2                                  2  2024-01-20T23:19:00+00:00  2024-01-20T23:20:00+00:00         3.409012e-02                  0.075291
1       carts-db         mem  8.053645e+07                          25                           2                                  2  2024-01-20T23:21:00+00:00  2024-01-20T23:22:00+00:00         8.068909e+07                  0.001895
2      catalogue  latency-90  4.837634e-03                          25                           2                                  2  2024-01-20T23:00:00+00:00  2024-01-20T23:01:00+00:00         4.895879e-03                  0.012040
3      catalogue         mem  6.132701e+06                          25                           2                                  2  2024-01-20T23:21:00+00:00  2024-01-20T23:22:00+00:00         6.137583e+06                  0.000796
4      front-end         cpu  4.979593e+00                          25                           2                                  2  2024-01-20T23:02:00+00:00  2024-01-20T23:03:00+00:00         4.989223e+00                  0.001934
5      front-end  latency-50  4.073165e-02                          25                           2                                  2  2024-01-20T23:19:00+00:00  2024-01-20T23:20:00+00:00         4.093088e-02                  0.004891
6      front-end      socket  2.124000e+01                          25                           2                                  2  2024-01-20T23:14:00+00:00  2024-01-20T23:15:00+00:00         2.160000e+01                  0.016949
7      orders-db         mem  4.091205e+07                          25                           2                                  2  2024-01-20T23:21:00+00:00  2024-01-20T23:22:00+00:00         4.099922e+07                  0.002131
8   queue-master      diskio  5.633100e+04                          25                           2                                  2  2024-01-20T23:11:00+00:00  2024-01-20T23:12:00+00:00         1.086807e+05                  0.929323
9   queue-master         mem  2.586175e+08                          25                           2                                  2  2024-01-20T23:13:00+00:00  2024-01-20T23:14:00+00:00         2.586318e+08                  0.000055
10      rabbitmq         mem  9.465504e+07                          25                           2                                  2  2024-01-20T23:21:00+00:00  2024-01-20T23:22:00+00:00         9.474185e+07                  0.000917
11      shipping         mem  3.033919e+08                          25                           2                                  2  2024-01-20T23:05:00+00:00  2024-01-20T23:06:00+00:00         3.034057e+08                  0.000045
12          user      diskio  2.548620e+04                          25                           2                                  2  2024-01-20T22:58:00+00:00  2024-01-20T22:59:00+00:00         8.215032e+04                  2.223325
13          user         mem  9.381595e+06                          25                           2                                  2  2024-01-20T23:21:00+00:00  2024-01-20T23:22:00+00:00         9.433566e+06                  0.005540
14          user      socket  3.595333e+01                          25                           2                                  2  2024-01-20T23:21:00+00:00  2024-01-20T23:22:00+00:00         3.609091e+01                  0.003827
15       user-db         cpu  1.147205e+00                          25                           2                                  2  2024-01-20T23:15:00+00:00  2024-01-20T23:16:00+00:00         1.150844e+00                  0.003172
16       user-db         mem  4.755939e+07                          25                           2                                  2  2024-01-20T23:21:00+00:00  2024-01-20T23:22:00+00:00         4.773379e+07                  0.003667
17         carts         cpu  4.298539e+00                          25                           2                                  1  2024-01-20T22:58:00+00:00  2024-01-20T23:06:00+00:00         1.647838e+01                  2.833484
18         carts  latency-50  1.613659e-02                          25                           2                                  1  2024-01-20T23:10:00+00:00  2024-01-20T23:19:00+00:00         1.657559e-02                  0.027205
19         carts         mem  2.064875e+08                          25                           2                                  1  2024-01-20T23:14:00+00:00  2024-01-20T23:20:00+00:00         2.068183e+08                  0.001602,               cmdb_id
0               carts
1            carts-db
2           catalogue
3        catalogue-db
4           front-end
5              orders
6           orders-db
7             payment
8        queue-master
9            rabbitmq
10  rabbitmq-exporter
11         session-db
12           shipping
13               user
14            user-db)```
```

In [ ]:
# In[3]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[3]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[4]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[4]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Load trace.csv (reuse trace_df if already in kernel)
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Parse timestamp to UTC datetime (needed for filtering)
if 'ts' not in trace_df.columns:
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Candidate services to analyze
candidates = ['carts', 'user', 'queue-master']

# 1) For each candidate list distinct (cmdb_id, trace_name) pairs and counts (limit 50)
pairs_count = (
    trace_df[trace_df['cmdb_id'].isin(candidates)]
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(50)
)

# 2) Compute global P95 for every (cmdb_id, trace_name) using the full trace series
global_p95_all = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_P95')
)

# Keep only global P95s for our candidates
global_p95 = global_p95_all[global_p95_all['cmdb_id'].isin(candidates)].copy()

# 3) Filter to incident window (UTC)
start_ts = pd.to_datetime("2024-01-20 22:55:32", utc=True)
end_ts = pd.to_datetime("2024-01-20 23:25:32", utc=True)
window_df = trace_df[
    (trace_df['ts'] >= start_ts) & (trace_df['ts'] <= end_ts) & (trace_df['cmdb_id'].isin(candidates))
].copy()

# 4) For each pair present in the window compute requested aggregates
# Merge window rows with global P95
window_df = window_df.merge(global_p95, on=['cmdb_id', 'trace_name'], how='left')

# Mark anomalies where value > global_P95
window_df['anomaly'] = window_df['value'] > window_df['global_P95']

def agg_group(g):
    cmdb = g.name[0]
    trace_name = g.name[1]
    gp95 = g['global_P95'].iloc[0]
    num_in_window = len(g)
    num_above = int(g['anomaly'].sum())
    if num_above == 0:
        earliest_iso = None
    else:
        earliest_iso = g.loc[g['anomaly'], 'ts'].min().isoformat()
    max_val = g['value'].max()
    # max_over_threshold_ratio handling gp95==0 -> inf
    if pd.isna(gp95):
        ratio = np.nan
    elif gp95 == 0:
        ratio = float('inf')
    else:
        ratio = (max_val - gp95) / gp95
    return pd.Series({
        'cmdb_id': cmdb,
        'trace_name': trace_name,
        'global_P95': gp95,
        'number_of_points_in_window': num_in_window,
        'number_of_points_above_P95': num_above,
        'earliest_timestamp_of_point_above_P95': earliest_iso,
        'max_value_in_window': max_val,
        'max_over_threshold_ratio': ratio
    })

if window_df.shape[0] > 0:
    agg_df = window_df.groupby(['cmdb_id', 'trace_name'], sort=False).apply(agg_group).reset_index(drop=True)
else:
    agg_df = pd.DataFrame(columns=[
        'cmdb_id','trace_name','global_P95','number_of_points_in_window',
        'number_of_points_above_P95','earliest_timestamp_of_point_above_P95',
        'max_value_in_window','max_over_threshold_ratio'
    ])

# 5) Return rows where number_of_points_above_P95>0 sorted by number_of_points_above_P95 desc then ratio desc, limit 20
anomalies = agg_df[agg_df['number_of_points_above_P95'] > 0].copy()
anomalies_sorted = anomalies.sort_values(
    by=['number_of_points_above_P95', 'max_over_threshold_ratio'],
    ascending=[False, False]
).head(20).reset_index(drop=True)

# Small list of distinct candidate cmdb_id values that had any trace anomalies
anomalous_cmdbs = pd.DataFrame({'cmdb_id': sorted(anomalies_sorted['cmdb_id'].unique())})

# Keep intermediate tables in kernel: trace_df, pairs_count, global_p95_all, global_p95, window_df, agg_df
# Display only requested compact outputs:
pairs_count, anomalies_sorted, anomalous_cmdbs

```
Out[5]:
```
Summary: I found no trace data for the three candidates (carts, user, queue-master) in trace.csv for the dataset and none within the incident window (2024-01-20 22:55:32 to 2024-01-20 23:25:32 UTC). Therefore:

- Distinct (cmdb_id, trace_name) pairs for those three services: none.
- No global P95s computed for those pairs (no series present).
- No trace points in the incident window for those services, so no trace anomalies detected.
- Because there are no trace records, I cannot confirm whether carts (or user / queue-master) appears downstream in traces during the anomaly times.

Next steps (if you want to continue):
- Verify that trace collection/instrumentation is enabled for these services and that trace.csv contains other services' traces.
- Broaden candidate list or inspect trace.csv contents to find related services that might show downstream evidence.
- Correlate with logs/metrics already showing anomalies (e.g., carts metric spikes) if traces are missing.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, trace_name, count]
Index: [], Empty DataFrame
Columns: [cmdb_id, trace_name, global_P95, number_of_points_in_window, number_of_points_above_P95, earliest_timestamp_of_point_above_P95, max_value_in_window, max_over_threshold_ratio]
Index: [], Empty DataFrame
Columns: [cmdb_id]
Index: [])```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse existing variables if present
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")

# Parse timestamps to UTC datetime if not already
if 'ts' not in log_df.columns:
    log_df['ts'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

# Incident window (UTC)
start_ts = pd.to_datetime("2024-01-20 22:55:32", utc=True)
end_ts = pd.to_datetime("2024-01-20 23:25:32", utc=True)

# Part A - log.csv

# 1) Distinct (cmdb_id, log_name) pairs with counts, sorted desc (limit to top 20 to keep outputs compact)
pairs_count = (
    log_df
    .groupby(['cmdb_id', 'log_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# 2) Global P95 per (cmdb_id, log_name) using full series (computed before any filtering)
global_p95_logs = (
    log_df
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_P95')
)

# 3) Filter to incident window
log_window = log_df[(log_df['ts'] >= start_ts) & (log_df['ts'] <= end_ts)].copy()

# Merge window with global P95
log_window = log_window.merge(global_p95_logs, on=['cmdb_id', 'log_name'], how='left')

# Mark anomalies where value > global_P95
log_window['anomaly'] = log_window['value'] > log_window['global_P95']

# Aggregate per (cmdb_id, log_name) in window
def agg_logs(g):
    cmdb = g.name[0]
    lname = g.name[1]
    gp95 = g['global_P95'].iloc[0]
    num_in_window = len(g)
    num_above = int(g['anomaly'].sum())
    if num_above == 0:
        earliest_iso = None
    else:
        earliest_iso = g.loc[g['anomaly'], 'ts'].min().isoformat()
    max_val = g['value'].max()
    return pd.Series({
        'cmdb_id': cmdb,
        'log_name': lname,
        'global_P95': gp95,
        'number_of_points_in_window': num_in_window,
        'number_of_points_above_P95': num_above,
        'earliest_timestamp_of_point_above_P95': earliest_iso,
        'max_value_in_window': max_val
    })

if log_window.shape[0] > 0:
    logs_agg = log_window.groupby(['cmdb_id', 'log_name'], sort=False).apply(agg_logs).reset_index(drop=True)
else:
    logs_agg = pd.DataFrame(columns=[
        'cmdb_id','log_name','global_P95','number_of_points_in_window',
        'number_of_points_above_P95','earliest_timestamp_of_point_above_P95','max_value_in_window'
    ])

# 4) Top anomalies: number_of_points_above_P95 > 0 sorted by number_of_points_above_P95 desc then max_value_in_window desc, limit 20
log_anomalies_top20 = (
    logs_agg[logs_agg['number_of_points_above_P95'] > 0]
    .sort_values(['number_of_points_above_P95', 'max_value_in_window'], ascending=[False, False])
    .head(20)
    .reset_index(drop=True)
)

# Part B - error_logs.csv (raw messages)
try:
    err_df
except NameError:
    err_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetime if not already
if 'ts' not in err_df.columns:
    err_df['ts'] = pd.to_datetime(err_df['timestamp'], unit='s', utc=True)

# 1) Filter error logs to incident window
err_window = err_df[(err_df['ts'] >= start_ts) & (err_df['ts'] <= end_ts)].copy()

# 2) Counts of messages per cmdb_id in the window (limit 20)
if err_window.shape[0] > 0:
    error_counts = (
        err_window
        .groupby('cmdb_id', dropna=False)
        .size()
        .reset_index(name='message_count')
        .sort_values('message_count', ascending=False)
        .head(20)
        .reset_index(drop=True)
    )
else:
    error_counts = pd.DataFrame(columns=['cmdb_id', 'message_count'])

# 3) Up to 20 sample raw messages in the window: timestamp (ISO UTC), cmdb_id, message sorted by timestamp asc
if err_window.shape[0] > 0:
    err_window['timestamp_iso'] = err_window['ts'].dt.tz_convert('UTC').apply(lambda x: x.isoformat())
    error_samples = err_window.sort_values('ts').loc[:, ['timestamp_iso', 'cmdb_id', 'message']].head(20).reset_index(drop=True)
else:
    error_samples = pd.DataFrame(columns=['timestamp_iso', 'cmdb_id', 'message'])

# Also concentrate on candidate services by filtering anomalies for candidates of interest
candidates = ['carts', 'user', 'queue-master', 'front-end']
log_candidate_anomalies = log_anomalies_top20[log_anomalies_top20['cmdb_id'].isin(candidates)].reset_index(drop=True)
error_candidate_counts = error_counts[error_counts['cmdb_id'].isin(candidates)].reset_index(drop=True)

# Display only compact aggregates (store intermediates in kernel): pairs_count, log_anomalies_top20, log_candidate_anomalies, error_counts, error_samples, error_candidate_counts
pairs_count, log_anomalies_top20, log_candidate_anomalies, error_counts, error_samples, error_candidate_counts

```
Out[6]:
```
Summary of log and error-log analysis for incident window 2024-01-20 22:55:32 → 2024-01-20 23:25:32 UTC:

Key findings (compact)
- Log KPI presence: many services emit log.row_count and log.error_count (counts per pair shown in the detailed table).
- Log anomalies (points above global P95) — top affected pairs:
  - front-end / log.row_count: P95=2046.0, 25 points in window, 2 points above P95, earliest above = 2024-01-20T22:59:00Z, max=2089
  - user / log.row_count: P95=620.2, 25 points, 2 above, earliest = 2024-01-20T22:59:00Z, max=625
  - catalogue / log.row_count: P95=170.6, 25 points, 2 above, earliest = 2024-01-20T23:02:00Z, max=174
  - carts / log.row_count: P95=123.75, 22 points, 2 above, earliest = 2024-01-20T22:59:00Z, max=150
  - queue-master / log.row_count: P95=585.0, 25 points, 1 above, earliest = 2024-01-20T22:59:00Z, max=594
  - (other services with single log.row_count anomalies: orders, payment, shipping)
- Note: log.error_count metrics exist for these services but did not appear in the top anomalies output (i.e., no strong error_count spikes above their global P95 in the window).

Error logs (raw messages)
- error_logs.csv returned no messages in this incident window (no raw error messages found).

Interpretation / action pointers
- Several candidate services (carts, user, front-end, queue-master) show small spikes in log.row_count around ~22:59–23:02 UTC; carts and user each had 2 anomalous points, queue-master had 1.
- No raw error messages were recorded in the error log file for the window, so investigate why increased log volume (row_count) occurred without corresponding error entries.
- Prioritize investigating carts and user (they show both metric and log anomalies) and front-end (largest absolute log-row spike) for root-cause tracing; if traces are missing, check instrumentation and correlate with metric spikes already observed.

If you want, I can (a) show the exact compact tables for these findings, (b) expand to other log KPI types, or (c) correlate these log-anomaly timestamps with the metric anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id         log_name  count
0      catalogue  log.error_count     25
1      catalogue    log.row_count     25
2         orders    log.row_count     25
3         orders  log.error_count     25
4      front-end    log.row_count     25
5      front-end  log.error_count     25
6       shipping    log.row_count     25
7       shipping  log.error_count     25
8           user    log.row_count     25
9           user  log.error_count     25
10       payment  log.error_count     25
11       payment    log.row_count     25
12  queue-master  log.error_count     25
13  queue-master    log.row_count     25
14      carts-db  log.error_count     24
15      carts-db    log.row_count     24
16         carts  log.error_count     22
17         carts    log.row_count     22
18    session-db  log.error_count      4
19    session-db    log.row_count      4,         cmdb_id       log_name  global_P95  number_of_points_in_window  number_of_points_above_P95 earliest_timestamp_of_point_above_P95  max_value_in_window
0     front-end  log.row_count     2046.00                          25                           2             2024-01-20T22:59:00+00:00                 2089
1          user  log.row_count      620.20                          25                           2             2024-01-20T22:59:00+00:00                  625
2     catalogue  log.row_count      170.60                          25                           2             2024-01-20T23:02:00+00:00                  174
3         carts  log.row_count      123.75                          22                           2             2024-01-20T22:59:00+00:00                  150
4  queue-master  log.row_count      585.00                          25                           1             2024-01-20T22:59:00+00:00                  594
5        orders  log.row_count      130.00                          25                           1             2024-01-20T22:59:00+00:00                  132
6       payment  log.row_count      105.00                          25                           1             2024-01-20T22:59:00+00:00                  106
7      shipping  log.row_count       65.00                          25                           1             2024-01-20T22:59:00+00:00                   66,         cmdb_id       log_name  global_P95  number_of_points_in_window  number_of_points_above_P95 earliest_timestamp_of_point_above_P95  max_value_in_window
0     front-end  log.row_count     2046.00                          25                           2             2024-01-20T22:59:00+00:00                 2089
1          user  log.row_count      620.20                          25                           2             2024-01-20T22:59:00+00:00                  625
2         carts  log.row_count      123.75                          22                           2             2024-01-20T22:59:00+00:00                  150
3  queue-master  log.row_count      585.00                          25                           1             2024-01-20T22:59:00+00:00                  594, Empty DataFrame
Columns: [cmdb_id, message_count]
Index: [], Empty DataFrame
Columns: [timestamp_iso, cmdb_id, message]
Index: [], Empty DataFrame
Columns: [cmdb_id, message_count]
Index: [])```
```